<div dir="rtl">

# الترم ٣ · الوحدة ٤: الذكاء الاصطناعي والإبداع

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-3/11-ai-creativity.ipynb)

إزاي الكمبيوتر "بيألّف" نص أو مزيكا أو رسم؟ الفكرة الأساسية: **يتعلم أنماط من أمثلة، وبعدين يولّد حاجة جديدة بنفس الأنماط**.

هنبني ٣ مولّدات صغيرة:
1. مولّد نصوص بـ **Markov chain**: الجد البعيد لـ ChatGPT.
2. مولّد ألحان.
3. فن توليدي (generative art).

</div>

In [ ]:
import random
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

<div dir="rtl">

## ١. مولّد النصوص: "الكلمة الجاية إيه؟"

الموديل بيعدّ: بعد كلمة X، الكلمات اللي جت بعدها كانت إيه وكام مرة. وبعدين بيولّد بإنه يختار الكلمة الجاية عشوائي حسب العدد ده.

**ده نفس سؤال ChatGPT بالظبط:** "إيه الكلمة الجاية؟". الفرق إن ChatGPT بيبص على آلاف الكلمات اللي فاتت بشبكة عصبية عملاقة، وإحنا بنبص على كلمة أو اتنين بس.

</div>

In [ ]:
CORPUS = """
the robot learns from data . the robot makes a plan . the robot finds the goal .
the student learns python . the student builds a robot . the student trains a model .
a model learns patterns from data . a model makes predictions . data helps the model learn .
the teacher explains the model . the teacher helps the student . the goal is to learn .
ai helps people . ai learns from examples . people build ai to help people .
"""

def train_markov(text: str, order: int = 1) -> dict:
    words = text.split()
    model = defaultdict(Counter)
    for i in range(len(words) - order):
        key = tuple(words[i:i + order])
        model[key][words[i + order]] += 1
    return model


def generate(model: dict, start: tuple, length: int = 15, seed: int | None = None) -> str:
    rng = random.Random(seed)
    out = list(start)
    for _ in range(length):
        options = model.get(tuple(out[-len(start):]))
        if not options:
            break
        words, counts = zip(*options.items())
        out.append(rng.choices(words, weights=counts)[0])
    return " ".join(out)

m1 = train_markov(CORPUS, order=1)
print("After 'the':", dict(m1[("the",)].most_common(5)))
print()
for s in range(3):
    print("-", generate(m1, ("the",), seed=s))

<div dir="rtl">

### الـ order بيفرق إزاي؟

</div>

In [ ]:
m2 = train_markov(CORPUS, order=2)
for s in range(3):
    print("-", generate(m2, ("the", "robot"), seed=s))

<div dir="rtl">

- **order=1:** إبداع أكتر بس جمل ملخبطة أحياناً.
- **order=2:** جمل منطقية أكتر بس بتنقل من الأصل كتير.

ده نفس التوازن في أي موديل توليدي: **الإبداع مقابل الدقة**. في ChatGPT فيه إعداد اسمه **temperature** بيتحكم في ده.

</div>

<div dir="rtl">

## ٢. Temperature: نتحكم في "الجرأة"

</div>

In [ ]:
def sample_with_temperature(counter: Counter, temperature: float, rng: np.random.Generator) -> str:
    words = list(counter)
    logits = np.log(np.array([counter[w] for w in words], dtype=float))
    probs = np.exp(logits / temperature)
    probs /= probs.sum()
    return words[rng.choice(len(words), p=probs)]

rng = np.random.default_rng(0)
after_the = m1[("the",)]
for t in [0.2, 1.0, 3.0]:
    picks = Counter(sample_with_temperature(after_the, t, rng) for _ in range(500))
    print(f"temperature={t}: {dict(picks.most_common(4))}")

<div dir="rtl">

حرارة **منخفضة** ← بيختار الكلمة الأشهر تقريباً دايماً (ممل بس آمن). حرارة **عالية** ← بيختار أي حاجة (مبدع بس ممكن يهبد).

</div>

<div dir="rtl">

## ٣. مولّد ألحان

نفس الفكرة على **نوتات** بدل كلمات. هنتعلم من لحن بسيط ونولّد لحن جديد ونرسمه.

</div>

In [ ]:
melody = "C C G G A A G F F E E D D C G G F F E E D G G F F E E D C C G G A A G F F E E D D C".split()
notes_order = ["C", "D", "E", "F", "G", "A", "B"]
note_model = train_markov(" ".join(melody), order=1)
new_melody = generate(note_model, ("C",), length=23, seed=10).split()
print("New melody:", " ".join(new_melody))

plt.figure(figsize=(10, 2.5))
plt.step(range(len(new_melody)), [notes_order.index(n) for n in new_melody], where="mid", color="#7C3AED")
plt.yticks(range(len(notes_order)), notes_order)
plt.xlabel("Time")
plt.title("Generated melody")
plt.show()

<div dir="rtl">

**جرّب تسمعها:** في Colab شغّل الخلية دي (بتعمل صوت لكل نوتة).

```python
from IPython.display import Audio
freqs = {"C": 261.6, "D": 293.7, "E": 329.6, "F": 349.2, "G": 392.0, "A": 440.0, "B": 493.9}
sr = 22050
t = np.linspace(0, 0.35, int(sr * 0.35), endpoint=False)
wave = np.concatenate([np.sin(2 * np.pi * freqs[n] * t) * np.exp(-3 * t) for n in new_melody])
Audio(wave, rate=sr)
```

</div>

<div dir="rtl">

## ٤. فن توليدي: قواعد بسيطة ← رسم معقد

</div>

In [ ]:
def generative_art(seed: int, n_lines: int = 300):
    rng = np.random.default_rng(seed)
    fig, ax = plt.subplots(figsize=(5, 5))
    angle, x, y = 0.0, 0.0, 0.0
    palette = plt.cm.plasma(np.linspace(0, 1, n_lines))
    for i in range(n_lines):
        angle += rng.normal(0, 0.6)
        step = 1 + i * 0.02
        nx, ny = x + step * np.cos(angle), y + step * np.sin(angle)
        ax.plot([x, nx], [y, ny], color=palette[i], linewidth=1.5, alpha=0.8)
        x, y = nx, ny
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"seed={seed}")
    plt.show()

generative_art(seed=7)

<div dir="rtl">

## 🏋️ تمرين ١: مولّد بجملك
اكتب corpus بنفسك (١٠ جمل على الأقل عن موضوع بتحبه: كورة، أكل، ألعاب) وولّد منه ٣ جمل.

<details><summary>الحل (مثال)</summary>

```python
football = """
salah scores a goal . salah runs fast . the team wins the match .
the fans love salah . the team trains hard . the coach loves the team .
the goal wins the match . the fans sing for the team . salah wins the cup .
the coach explains the plan .
"""
fm = train_markov(football, order=1)
for s in range(3):
    print(generate(fm, ("salah",), seed=s))
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🤔 نقاش: مين صاحب العمل الفني؟

1. لو موديل اتعلم من رسومات فنانين من غير إذنهم، ده عادل؟
2. لو عملت رسمة بالذكاء الاصطناعي، تقول إنك إنت اللي رسمتها؟
3. إزاي نفرّق بين صورة حقيقية وصورة متولدة (deepfake)؟ ليه ده مهم؟

## ✅ لخّص
- الموديلات التوليدية = تتعلم أنماط ← تولّد جديد بنفس الأنماط.
- **Markov chain** = "الكلمة الجاية إيه؟" بأبسط شكل.
- **Temperature** = التحكم في الجرأة/الإبداع.
- مع الإبداع فيه مسؤولية: حقوق الفنانين، الأمانة، والـ deepfakes.

</div>